## Intro to Env

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

import numpy as np
import gymnasium as gym
from gymnasium.spaces import Discrete, Box



In [5]:
env = gym.make("LunarLander-v3", render_mode="human")

obs, info = env.reset(seed=42)

for _ in range(1000):
    action = env.action_space.sample()

    next_obs, rew, terminated, truncated, info = env.step(action)

    if terminated or truncated:
        observation, info = env.reset()
env.close()


In [11]:
env = gym.make("CartPole-v1", render_mode="human")

obs, info = env.reset()

print("Starting point", obs)
# [cart_position, cart_velocity, pole_angle, pole_angular_velocity]

episode_over = False
total_reward = 0

while not episode_over:
    # Sample action from policy
    action = env.action_space.sample()

    # make step and get new_obs, reward, terminated, truncated, info
    new_obs, reward, terminated, truncated, info = env.step(action)

    total_reward += reward

    episode_over = terminated or truncated

env.close()

print(f"Episode finished! Total reward: {total_reward}")

Starting point [-0.0153087   0.01176756  0.0032856  -0.04051443]
Episode finished! Total reward: 18.0


In [33]:
print(env.observation_space)
print(env.action_space)
print(f"Sample observation: {env.observation_space.sample()}")

Box([-4.8               -inf -0.41887903        -inf], [4.8               inf 0.41887903        inf], (4,), float32)
Discrete(2)
Sample observation: [ 3.9618988   0.58536786 -0.10076723 -0.13115439]


In [34]:
env.action_space

Discrete(2)

In [80]:
from torch.distributions.categorical import Categorical

Categorical(logits=torch.tensor([0.2, 100])).sample()

tensor(1)

## Vanilla Policy Gradient

From SpinningUP course

In [ ]:
from torch.distributions.categorical import Categorical

def mlp(sizes, inter_act=nn.Tanh, output_act=nn.Identity):
    layers = []
    for i in range(len(sizes)-1):
        act = inter_act if i < len(sizes) - 2 else output_act
        layers.extend([nn.Linear(sizes[i], sizes[i+1]), act()])
    return nn.Sequential(*layers)


def reward_to_go(rews):
    n = len(rews)
    rtgs = np.zeros_like(rews)
    for i in reversed(range(n)):
        rtgs[i] = rews[i] + (rtgs[i+1] if i+1 < n else 0)
    return rtgs


def train(env_name: str, hidden_size=[32], lr=1e-2, epochs=50, batch_size=5000, render=False, use_reward_to_go=False):
    # create env
    env = gym.make(env_name)
    
    # for now we focus only on continious obs spaces and discrete action spaces
    assert isinstance(env.observation_space, Box), \
        "This example only works for envs with continuous state spaces."
    assert isinstance(env.action_space, Discrete), \
        "This example only works for envs with discrete action spaces."
    
    # Get obs dim and action dim.
    obs_dim = env.observation_space.shape[0]
    n_acts = env.action_space.n

    # Create Policy
    policy = mlp([obs_dim] + hidden_size + [n_acts])
    optimizer = optim.Adam(policy.parameters(), lr=lr)

    def policy_grad_loss(obs, action, rets):
        log_probs = Categorical(logits=policy(torch.as_tensor(obs, dtype=torch.float32))).log_prob(action)
        return -(log_probs * rets).mean()
    
    def train_epoch():
        # In order to calculate Policy Gradient we need to save full trajectory: s_i, a_i, s_i+1, r_i
        batch_obs = []
        batch_action = []
        batch_rets = []

        episodes_rets = []         # for measuring episode returns
        episodes_lens = []         # for measuring episode lengths
        # we play the game using our policy until we get batch_size amount of obs.

        obs, _ = env.reset()
        eps_rewards = []

        # render first episode of each epoch
        finished_rendering_this_epoch = False

        while True:

            # rendering
            if (not finished_rendering_this_epoch) and render:
                env.render()

            # chose action based on current obs and our policy
            action = Categorical(logits=policy(torch.as_tensor(obs, dtype=torch.float32))).sample().item()

            # log obs and action
            batch_obs.append(obs.copy())
            batch_action.append(action)

            # take action:
            obs, reward, terminated, truncated, info = env.step(action)

            # log the reward
            eps_rewards.append(reward)

            # check if episode over
            if truncated or terminated:
                eps_return, eps_len = sum(eps_rewards), len(eps_rewards)

                episodes_rets.append(eps_return)
                episodes_lens.append(eps_len)

                if use_reward_to_go:
                    batch_rets.extend(reward_to_go(eps_rewards))
                else:
                    batch_rets.extend([eps_return]*eps_len)

                obs, _ = env.reset()
                eps_rewards = []

                finished_rendering_this_epoch = True

                if len(batch_obs) > batch_size:
                    break
    
        optimizer.zero_grad()
        loss = policy_grad_loss(obs=torch.as_tensor(batch_obs, dtype=torch.float32), 
                                action=torch.as_tensor(batch_action, dtype=torch.int64),
                                rets=torch.as_tensor(batch_rets, dtype=torch.float32))
        loss.backward()
        optimizer.step()

        return loss, episodes_rets, episodes_lens
    
    for epoch in range(epochs):
        loss, episodes_rets, episodes_lens = train_epoch()
        print('epoch: %3d \t loss: %.3f \t return: %.3f \t ep_len: %.3f'%
                (epoch, loss, np.mean(episodes_rets), np.mean(episodes_lens)))

# if __name__ == '__main__':
#     import argparse
#     parser = argparse.ArgumentParser()
#     parser.add_argument('--env_name', '--env', type=str, default='CartPole-v0')
#     parser.add_argument('--render', action='store_true')
#     parser.add_argument('--lr', type=float, default=1e-2)
#     args = parser.parse_args()
#     print('\nUsing simplest formulation of policy gradient.\n')
#     train(env_name=args.env_name, render=args.render, lr=args.lr)

        

            
        


In [136]:
train(env_name="LunarLander-v3", use_reward_to_go=False)

epoch:   0 	 loss: -296.797 	 return: -222.252 	 ep_len: 105.896
epoch:   1 	 loss: -272.069 	 return: -197.174 	 ep_len: 103.286
epoch:   2 	 loss: -275.073 	 return: -196.923 	 ep_len: 96.885
epoch:   3 	 loss: -345.531 	 return: -247.882 	 ep_len: 96.962
epoch:   4 	 loss: -170.091 	 return: -158.751 	 ep_len: 110.674
epoch:   5 	 loss: -208.687 	 return: -147.915 	 ep_len: 94.887
epoch:   6 	 loss: -299.303 	 return: -209.170 	 ep_len: 94.148
epoch:   7 	 loss: -231.066 	 return: -168.445 	 ep_len: 88.088
epoch:   8 	 loss: -228.308 	 return: -158.435 	 ep_len: 90.143
epoch:   9 	 loss: -185.404 	 return: -178.150 	 ep_len: 107.319
epoch:  10 	 loss: -184.815 	 return: -140.509 	 ep_len: 83.950
epoch:  11 	 loss: -173.837 	 return: -131.151 	 ep_len: 85.186
epoch:  12 	 loss: -142.313 	 return: -146.348 	 ep_len: 106.894
epoch:  13 	 loss: -175.658 	 return: -134.318 	 ep_len: 83.750
epoch:  14 	 loss: -177.938 	 return: -136.495 	 ep_len: 91.236
epoch:  15 	 loss: -160.101 	 retur

In [139]:
train(env_name="LunarLander-v3", use_reward_to_go=True)

epoch:   0 	 loss: -210.624 	 return: -198.444 	 ep_len: 85.932
epoch:   1 	 loss: -181.700 	 return: -166.499 	 ep_len: 79.453
epoch:   2 	 loss: -179.084 	 return: -169.734 	 ep_len: 78.688
epoch:   3 	 loss: -152.760 	 return: -149.888 	 ep_len: 74.382
epoch:   4 	 loss: -147.119 	 return: -141.610 	 ep_len: 76.379
epoch:   5 	 loss: -142.131 	 return: -138.809 	 ep_len: 77.231
epoch:   6 	 loss: -138.207 	 return: -134.136 	 ep_len: 73.662
epoch:   7 	 loss: -130.496 	 return: -130.788 	 ep_len: 78.141
epoch:   8 	 loss: -140.760 	 return: -140.750 	 ep_len: 75.045
epoch:   9 	 loss: -136.947 	 return: -143.802 	 ep_len: 76.227
epoch:  10 	 loss: -130.739 	 return: -133.475 	 ep_len: 76.258
epoch:  11 	 loss: -119.604 	 return: -129.047 	 ep_len: 75.848
epoch:  12 	 loss: -117.593 	 return: -124.177 	 ep_len: 72.986
epoch:  13 	 loss: -125.836 	 return: -135.160 	 ep_len: 71.586
epoch:  14 	 loss: -128.848 	 return: -138.863 	 ep_len: 73.838
epoch:  15 	 loss: -127.737 	 return: -1

: 

## Vanilla Policy Gradient SpinningUP Implementation 

In [127]:
import torch
import torch.nn as nn
from torch.distributions.categorical import Categorical
from torch.optim import Adam
import numpy as np
import gymnasium as gym
from gymnasium.spaces import Discrete, Box

def mlp(sizes, activation=nn.Tanh, output_activation=nn.Identity):
    # Build a feedforward neural network.
    layers = []
    for j in range(len(sizes)-1):
        act = activation if j < len(sizes)-2 else output_activation
        layers += [nn.Linear(sizes[j], sizes[j+1]), act()]
    return nn.Sequential(*layers)

def train(env_name='CartPole-v0', hidden_sizes=[32], lr=1e-2, 
          epochs=50, batch_size=5000, render=False):

    # make environment, check spaces, get obs / act dims
    env = gym.make(env_name)
    assert isinstance(env.observation_space, Box), \
        "This example only works for envs with continuous state spaces."
    assert isinstance(env.action_space, Discrete), \
        "This example only works for envs with discrete action spaces."

    obs_dim = env.observation_space.shape[0]
    n_acts = env.action_space.n

    # make core of policy network
    logits_net = mlp(sizes=[obs_dim]+hidden_sizes+[n_acts])

    # make function to compute action distribution
    def get_policy(obs):
        logits = logits_net(obs)
        return Categorical(logits=logits)

    # make action selection function (outputs int actions, sampled from policy)
    def get_action(obs):
        return get_policy(obs).sample().item()

    # make loss function whose gradient, for the right data, is policy gradient
    def compute_loss(obs, act, weights):
        logp = get_policy(obs).log_prob(act)
        return -(logp * weights).mean()

    # make optimizer
    optimizer = Adam(logits_net.parameters(), lr=lr)

    # for training policy
    def train_one_epoch():
        # make some empty lists for logging.
        batch_obs = []          # for observations
        batch_acts = []         # for actions
        batch_weights = []      # for R(tau) weighting in policy gradient
        batch_rets = []         # for measuring episode returns
        batch_lens = []         # for measuring episode lengths

        # reset episode-specific variables
        obs, _ = env.reset()       # first obs comes from starting distribution
        done = False            # signal from environment that episode is over
        ep_rews = []            # list for rewards accrued throughout ep

        # render first episode of each epoch
        finished_rendering_this_epoch = False

        # collect experience by acting in the environment with current policy
        while True:

            # rendering
            if (not finished_rendering_this_epoch) and render:
                env.render()

            # save obs
            batch_obs.append(obs.copy())

            # act in the environment
            act = get_action(torch.as_tensor(obs, dtype=torch.float32))
            obs, rew, terminated, truncated, _ = env.step(act)

            # save action, reward
            batch_acts.append(act)
            ep_rews.append(rew)

            if terminated or truncated:
                # if episode is over, record info about episode
                ep_ret, ep_len = sum(ep_rews), len(ep_rews)
                batch_rets.append(ep_ret)
                batch_lens.append(ep_len)

                # the weight for each logprob(a|s) is R(tau)
                batch_weights += [ep_ret] * ep_len

                # reset episode-specific variables
                obs, _ = env.reset()
                done, ep_rews = False, []

                # won't render again this epoch
                finished_rendering_this_epoch = True

                # end experience loop if we have enough of it
                if len(batch_obs) > batch_size:
                    break

        # take a single policy gradient update step
        optimizer.zero_grad()
        batch_loss = compute_loss(obs=torch.as_tensor(batch_obs, dtype=torch.float32),
                                  act=torch.as_tensor(batch_acts, dtype=torch.int32),
                                  weights=torch.as_tensor(batch_weights, dtype=torch.float32)
                                  )
        batch_loss.backward()
        optimizer.step()
        return batch_loss, batch_rets, batch_lens

    # training loop
    for i in range(epochs):
        batch_loss, batch_rets, batch_lens = train_one_epoch()
        print('epoch: %3d \t loss: %.3f \t return: %.3f \t ep_len: %.3f'%
                (i, batch_loss, np.mean(batch_rets), np.mean(batch_lens)))

In [128]:
train(env_name="LunarLander-v3")

epoch:   0 	 loss: -322.846 	 return: -228.652 	 ep_len: 100.880
epoch:   1 	 loss: -323.762 	 return: -232.672 	 ep_len: 96.750
epoch:   2 	 loss: -251.492 	 return: -177.788 	 ep_len: 93.167
epoch:   3 	 loss: -236.877 	 return: -162.296 	 ep_len: 90.429
epoch:   4 	 loss: -207.570 	 return: -146.760 	 ep_len: 90.286
epoch:   5 	 loss: -218.562 	 return: -155.357 	 ep_len: 90.089
epoch:   6 	 loss: -223.134 	 return: -160.418 	 ep_len: 81.452
epoch:   7 	 loss: -205.670 	 return: -143.918 	 ep_len: 82.279
epoch:   8 	 loss: -183.024 	 return: -132.061 	 ep_len: 82.705
epoch:   9 	 loss: -180.690 	 return: -134.873 	 ep_len: 85.475
epoch:  10 	 loss: -175.291 	 return: -131.739 	 ep_len: 81.048
epoch:  11 	 loss: -169.880 	 return: -123.347 	 ep_len: 80.806
epoch:  12 	 loss: -155.676 	 return: -116.195 	 ep_len: 77.385
epoch:  13 	 loss: -176.711 	 return: -131.482 	 ep_len: 82.066
epoch:  14 	 loss: -156.493 	 return: -117.529 	 ep_len: 78.891
epoch:  15 	 loss: -180.496 	 return: -